In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from rlm_sec.filings import sec_data
# from rlm_sec.filings.utils import company_to_ticker
# import asyncio

# from rlm_sec.trainer import hf_dataloader

# all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]
# all_tickers_years = list(set(all_tickers_years))
# combined_qa = hf_dataloader.load_combined_qa()

# # Function to call sec_main for a given (ticker, year)
# def fetch_sec_main(args):
#     ticker, year = args
#     ticker = company_to_ticker(ticker)
#     # sec_main is async, so run it with asyncio
#     if not ticker:
#         return None, None, None
#     return (ticker, year, asyncio.run(sec_data.sec_main(ticker, year)))

# results = []
# with ThreadPoolExecutor() as executor:
#     # Submit all (ticker, year) pairs for execution
#     futures = [executor.submit(fetch_sec_main, (ticker, year)) for ticker, year in all_tickers_years]
#     for future in as_completed(futures):
#         try:
#             ticker, year, value = future.result()
#             if not ticker:
#                 continue
#             results.append((ticker, year, value))
#         except Exception as e:
#             print(f"Error fetching ({ticker}, {year}): {e}")


In [ ]:
# import re
# from pathlib import Path
# from settings import env_settings
# from rlm_sec.dataloader.vector_store import FaissVectorIndex

# # Root containing one directory per "TICKER-YYYY" with *.md inside each.
# MARKDOWN_ROOT = Path("localworkspace/markdown/sec_data/")
# FORCE_REBUILD = False

# # Directory names must end with -YYYY (handles tickers like BRK-B-2025).
# _TICKER_YEAR_DIR = re.compile(r"^(?P<ticker>.+)-(?P<year>\d{4})$")

# index = FaissVectorIndex()
# all_keys = []

# for sub in sorted(MARKDOWN_ROOT.iterdir()):
#     if not sub.is_dir():
#         continue
#     m = _TICKER_YEAR_DIR.match(sub.name)
#     if not m:
#         print(f"skip (not TICKER-YYYY): {sub.name}")
#         continue
#     ticker, year = m["ticker"], m["year"]
#     md_paths = sorted(sub.glob("*.md"))
#     if not md_paths:
#         print(f"skip (no .md): {sub.name}")
#         continue
#     try:
#         keys = index.from_markdown(
#             ticker=ticker,
#             year=year,
#             markdown_paths=md_paths,
#             force=True,
#         )
#     except Exception:
#         pass
#     all_keys.extend(keys)
#     print(f"indexed {sub.name}: {len(keys)} filing(s)")

# print(f"total index keys: {len(all_keys)}")

## TESTING ENVIRONMENT

In [1]:
from rlm_sec.trainer import hf_dataloader

combined_qa = hf_dataloader.load_combined_qa()
# all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]
# all_tickers_years = list(set(all_tickers_years))

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 150/150 [00:00<00:00, 6635.37 examples/s]


In [2]:
print(combined_qa[-200]['prompt'][-1]['content'])

Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you lack some knowledge, you can call one of the two search tools below.

Tool 1 — SEC Filings (annual and quarterly reports):
  <search>SECFilingTool(query, ticker, year, filing_type)</search>
  filing_type is one of: 10-K (annual), 10-Q1, 10-Q2, 10-Q3 (quarterly).
  Example: <search>SECFilingTool(cash flow from operations, AAPL, 2023, 10-K)</search>

Tool 2 — Earnings Call Transcripts:
  <search>EarningsTranscriptTool(query, ticker, year, quarter)</search>
  quarter is one of: Q1, Q2, Q3, Q4.
  Example: <search>EarningsTranscriptTool(cash flow from operations, MSFT, 2023, Q2)</search>

The search engine will return results between <information> and </information>. You can search as many times as needed. Once you have sufficient information, provide the final answer inside <answer> and </answer> without additional explanation. For example, <an

In [6]:
cnt = 0
for i in combined_qa:
    if i['data_source'] != 'virattt/financial-qa-10K':
        # print(i['prompt'][-1]['content'])
        cnt += 1
print(cnt)


150


In [8]:
import re

for i in combined_qa:
    if i['data_source'] != 'virattt/financial-qa-10K':
        content = i['prompt'][-1]['content']
        m = re.search(r"Question:(.*)", content, re.DOTALL)
        if m:
            print(m.group(1).strip())


What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
Is 3M a capital-intensive business based on FY2022 data?
What drove operating margin change as of FY2022 for 3M? If operating margin is not a useful metric for a company like this, then please state that and explain why.
If we exclude the impact of M&A, which segment has dragged down 3M's overall growth in 2022?
Does 3M have a reasonably healthy liquidity profile based on its quick ratio for Q2 of FY2023? If the quick ratio is not relevant to measure liquidity, please state that and explain why.
Which debt securities are registered to trade on a national securities exchange under 3M's name as of Q2 of 2023?
Does 3M 

In [ ]:
from datasets import load_dataset

# Load the "validation.parquet" file from the local directory using HuggingFace datasets
dataset = load_dataset("parquet", data_files="data/searchR1/validation.parquet")["train"]

dataset

Generating train split: 51713 examples [00:00, 72419.97 examples/s]

Dataset({
    features: ['data_source', 'prompt', 'ability', 'env_class', 'reward_spec', 'extra_info', 'metadata'],
    num_rows: 51713
})


In [16]:
dataset[0]

{'data_source': 'searchR1_nq',
 'prompt': [{'content': 'You are a helpful and harmless assistant.',
   'role': 'system'},
  {'content': 'Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. You can search as many times as you want. If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer>, without detailed illustrations. For example, <answer> Beijing </answer>. Question: who got the first nobel prize in physics?',
   'role': 'user'}],
 'ability': 'fact-reasoning',
 'env_class': 'search',
 'reward_spec': {'ground_truth': {'target': ['Wilhelm Conrad Röntgen']},
  'style': 'rule'},
 'extra_info': {'index': 0,
  'need_tools_kwargs': True,
  'question': 'who got the firs

In [2]:
import requests
from settings import env_settings

url = f"{env_settings.server_url}/vector_store/search"
r = requests.post(
    url,
    json={
        "ticker": "AAPL",
        "year": "2023",
        "filing_type": "10-K",
        "query": "revenue recognition policy",
        "top_k": 3,
    },
    timeout=30,
)
r.raise_for_status()
chunks = r.json()
# for c in chunks:
#     print(c.get("score"), c.get("text", ""))
chunks

[{'text': 'Other Current Liabilities',
  'chunk_type': 'text',
  'page_num': 78,
  'section_title': 'Item 8. Financial Statements and Supplementary Data',
  'chunk_index': 150,
  'score': 0.5170146822929382},
 {'text': 'Other Non-Current Liabilities',
  'chunk_type': 'text',
  'page_num': 78,
  'section_title': 'Item 8. Financial Statements and Supplementary Data',
  'chunk_index': 152,
  'score': 0.5166018009185791},
 {'text': 'The Company has identified up to three performance obligations regularly included in arrangements involving the sale of iPhone, Mac, iPad and certain other products. The first performance obligation, which represents the substantial portion of the allocated sales price, is the hardware and bundled software delivered at the time of sale. The second performance obligation is the right to receive certain product-related bundled services, which include iCloud®, Siri® and Maps. The third performance obligation is the right to receive, on a when-and-if-available basi

In [1]:
import json

from rlm_sec.envs.finance_env import FinanceSearchEnv, SearchEnvConfig
from settings import env_settings

cfg = SearchEnvConfig(
    topk=3,
    timeout=30,
    log_requests=True,
)

extras = {"max_turns": 2}
env = FinanceSearchEnv(cfg, extras=extras)

# Required for reward on terminal step (not set in SECSearchEnv today):
env.ground_truth = {"target": "some canonical answer string"}

# system_user_messages, meta = env.init(
#     [
#         {"role": "user", "content": "Answer the question using search. Wrap queries in <search>...</search> and the final answer in <answer>...</answer>."},
#     ]
# )

# Simulate first model turn: search
out1 = env.step("<search>SECFilingTool(cash flow from operations, AAPL, 2023, 10-Q3)</search>")
print("done:", out1["done"], "reward:", out1["reward"])
print("metadata:", out1["metadata"])
if out1["observations"]:
    obs = out1["observations"][0]["content"]
    print("observation (prefix):", obs[:1500])

# Simulate second turn: final answer (ends episode)
out2 = env.step("<answer>some canonical answer string</answer>")
print("done:", out2["done"], "reward:", out2["reward"])

done: False reward: RewardType(correctness=0.0, format=1.6666666666666665)
metadata: {'tool_group': 'SECFilingToolGroup', 'tool_name': 'sec_filing_to_markdown_embed_and_search', 'tool_input': ParsedSearch(action='<search>SECFilingTool(cash flow from operations, AAPL, 2023, 10-Q3)</search>', query='cash flow from operations', ticker='AAPL', year='2023', filing_type_or_quarter='10-Q3', tool_group_name='SECFilingToolGroup', tool_name='sec_filing_to_markdown_embed_and_search', task_type='sec_filings'), 'tool_metadata': {'query': '', 'ticker': 'AAPL', 'year': '2023', 'filing_type': '10-Q3', 'api_request_error': None, 'api_response': [{'text': 'Property, Plant and Equipment, Net', 'chunk_type': 'text', 'page_num': 23, 'section_title': 'PART I — FINANCIAL INFORMATION', 'index': 42}, {'text': 'Item 6.  Exhibits', 'chunk_type': 'text', 'page_num': 45, 'section_title': 'Item 6.  Exhibits', 'index': 92}, {'text': 'None.', 'chunk_type': 'text', 'page_num': 44, 'section_title': 'Item 3. Defaults Up

In [1]:
import json
from rlm_sec.envs.finance_env import FinanceSearchEnv, SearchEnvConfig

cfg = SearchEnvConfig(topk=3, timeout=30, log_requests=True)
env = FinanceSearchEnv(cfg, extras={"max_turns": 4})
env.ground_truth = {"target": "some canonical answer string"}

# ── Turn 1: SEC filing search ─────────────────────────────────────────────────
out1 = env.step(
    "<search>SECFilingTool(cash flow from operations, AAPL, 2023, 10-Q3)</search>"
)
print(f"Turn 1 | done={out1['done']} reward={out1['reward']}")
print(f"  parse reward  : {out1['metadata'].get('tool_input') and out1['metadata']['tool_input'].reward}")
print(f"  observation   : {out1['observations'][0]['content'][:300] if out1['observations'] else '(none)'}")

# ── Turn 2: Earnings transcript search ───────────────────────────────────────
out2 = env.step(
    "<search>EarningsTranscriptTool(capital expenditure guidance, AAPL, 2023, Q3)</search>"
)
print(f"\nTurn 2 | done={out2['done']} reward={out2['reward']}")
print(f"  observation   : {out2['observations'][0]['content'][:300] if out2['observations'] else '(none)'}")

# ── Turn 3: Malformed search (tests partial reward) ───────────────────────────
out3 = env.step(
    "<search>SECFilingTool(missing args, AAPL)</search>"
)
print(f"\nTurn 3 | done={out3['done']} reward={out3['reward']}")
print(f"  observation   : {out3['observations'][0]['content'][:300] if out3['observations'] else '(none)'}")

# ── Turn 4: Final answer (terminates episode) ─────────────────────────────────
out4 = env.step("<answer>some canonical answer string</answer>")
print(f"\nTurn 4 | done={out4['done']} reward={out4['reward']}")

Turn 1 | done=False reward=RewardType(correctness=0.0, format=1.6666666666666665)
  parse reward  : 1.6666666666666665
  observation   : 
<information>
Doc 1: Property, Plant and Equipment, Net
Doc 2: Item 6.  Exhibits
Doc 3: None.
</information>


Turn 2 | done=False reward=RewardType(correctness=0.0, format=1.6666666666666665)
  observation   : 
<information>
Doc 1: <speaker-end>
Doc 2: <speaker-end>
Doc 3: <speaker-start>
### Operator

Our next question is from David Vogt with UBS.
<speaker-end>
</information>


Turn 3 | done=False reward=RewardType(correctness=0.0, format=0.0)
  observation   : 
<information>Invalid <search> format. Expected: SECFilingTool(ticker, year, filing_type) or EarningsTranscriptTool(ticker, year, quarter).</information>
But got: <search>SECFilingTool(missing args, AAPL)</search>

Turn 4 | done=True reward=RewardType(correctness=1.0, format=1.0)


In [3]:
import json
with open("document_ranking_kaggle_dev.jsonl", "r") as f:
    first_line = f.readline()
    first_data = json.loads(first_line)

In [7]:
print(first_data['messages'][0]['content'])

Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.

Question: How has Agilent Technologies’ instrument reliability metric for its core diagnostics manufacturing process changed recently?

Document Types to rank:
[Document Index 0] DEF14A

[Document Index 1] 10-K

[Document Index 2] 10-Q

[Document Index 3] 8-K

[Document Index 4] Earnings

Your response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index.


In [6]:
first_data

{'uuid': 'qe100cdf8e8f5',
 'messages': [{'role': 'user',
   'content': 'Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.\n\nQuestion: How has Agilent Technologies’ instrument reliability metric for its core diagnostics manufacturing process changed recently?\n\nDocument Types to rank:\n[Document Index 0] DEF14A\n\n[Document Index 1] 10-K\n\n[Document Index 2] 10-Q\n\n[Document Index 3] 8-K\n\n[Document Index 4] Earnings\n\nYour response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index.'}],
 'qrel': {'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}}

In [9]:
q = {"_id": "doc_q5c563e", "messages": [{"role": "user", "content": "Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.\n\nQuestion: What view did leadership express on competition from regional theme parks and resorts\n\nDocument Types to rank:\n[Document Index 0] DEF14A\n\n[Document Index 1] 10-K\n\n[Document Index 2] 10-Q\n\n[Document Index 3] 8-K\n\n[Document Index 4] Earnings\n\nYour response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index."}]}

q.keys()

dict_keys(['_id', 'messages'])

In [24]:
import pandas as pd

train_df = pd.read_parquet("data/train.parquet")
val_df = pd.read_parquet("data/validation.parquet")

In [28]:
train_df.shape

(10922, 11)

In [29]:
val_df.shape

(1214, 11)

In [30]:
len(train_df)

10922

In [31]:
len(val_df)

1214